In [ ]:
import os
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[var] = "1"
from joblib import Parallel, delayed

import numpy as np

from dysts.maps import Henon

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error

from spring_rc.v1 import (
    build_von_mises_lattice,
    run_simulation,
    scatter_plot,
    save_figure,
    r_2_plots_grid,
)

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [3]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [4]:
num_cells = 100
cell_spacing = 2.5
wall_dist = 1.0
mass_val = 0.01
damping_val = 0.2
k_wall_val = 5.0
k_inter_val = 4.0
rest_wall_ratio = 1.1
rest_inter_ratio = 1.1
k_wall_spread = 1.0
k_inter_spread = 1.0
rest_wall_ratio_spread = 0.0
rest_inter_ratio_spread = 0.0

dt = 0.01
target_node_count = 10
input_force = 10
free_steps = 3

N_step = int(num_cells / target_node_count)
target_nodes = 1 + (np.arange(target_node_count) * N_step) * 3

F_in = np.zeros((total_steps_after_chaos, target_node_count, 2))
for j in range(target_nodes.shape[0]):
    F_in[::, j, 0] = henon_scaled[:, j % 2]
F_in = F_in * input_force

In [5]:
def seeded_trial(seed):
    rng = np.random.default_rng(seed)

    (
        init_pos,
        m_inv_diag,
        c_diag,
        connections_list,
        wall_nodes,
        k_vals,
        rest_lens,
    ) = build_von_mises_lattice(
        rng=rng,
        num_cells=num_cells,
        cell_spacing=cell_spacing,
        wall_dist=wall_dist,
        mass_val=mass_val,
        damping_val=damping_val,
        k_wall_val=k_wall_val,
        k_wall_spread=k_wall_spread,
        k_inter_val=k_inter_val,
        k_inter_spread=k_inter_spread,
        rest_wall_ratio=rest_wall_ratio,
        rest_wall_ratio_spread=rest_wall_ratio_spread,
        rest_inter_ratio=rest_inter_ratio,
        rest_inter_ratio_spread=rest_inter_ratio_spread,
    )

    Xd, _ = run_simulation(
        steps=total_steps_after_chaos,
        free_steps=free_steps,
        dt=dt,
        m_inv=m_inv_diag,
        c_diag=c_diag,
        F_in=F_in,
        target_nodes=target_nodes,
        init_pos=init_pos,
        conn=connections_list,
        k=k_vals,
        rest=rest_lens,
        fixed_nodes=wall_nodes,
        rec_nodes=1 + 3 * np.arange(num_cells),
    )

    X_data = Xd[transient_steps_reservoir:-tau_steps]
    Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]
    x_scaler = StandardScaler()
    X_train, X_test = (
        x_scaler.fit_transform(X_data[:-test_steps]),
        x_scaler.transform(X_data[-test_steps:]),
    )
    Y_train, Y_test_unscaled = (
        Y_data[:-test_steps],
        Y_data[-test_steps:],
    )
    model = RidgeCV()
    model.fit(X_train, Y_train)
    Y_pred_scaled = model.predict(X_test)
    Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
    Y_test = henon_scaler.inverse_transform(Y_test_unscaled)

    rmse = root_mean_squared_error(Y_test, Y_pred, multioutput="raw_values")
    nrmse = rmse / np.std(Y_test, axis=0)
    r_2 = r2_score(Y_test, Y_pred, multioutput="raw_values")

    return seed, nrmse, r_2

In [6]:
seeds = np.arange(100)

In [7]:
results = [seeded_trial(seed) for seed in seeds]
for seed, nrmse, r_2 in results[::10]:
    print(f"Seed {seed}: NRMSE = {nrmse}, R² = {r_2}")

Seed 0: NRMSE = [0.84961985 0.04403428], R² = [0.27814611 0.99806098]
Seed 10: NRMSE = [0.85448154 0.06008886], R² = [0.2698613  0.99638933]
Seed 20: NRMSE = [0.85304171 0.08463523], R² = [0.27231983 0.99283688]
Seed 30: NRMSE = [0.84546227 0.02583743], R² = [0.28519355 0.99933243]
Seed 40: NRMSE = [0.84929764 0.04147623], R² = [0.27869352 0.99827972]
Seed 50: NRMSE = [0.84213403 0.01713274], R² = [0.29081027 0.99970647]
Seed 60: NRMSE = [0.84447049 0.01510623], R² = [0.28686959 0.9997718 ]
Seed 70: NRMSE = [0.84462873 0.05503047], R² = [0.28660231 0.99697165]
Seed 80: NRMSE = [0.84024171 0.02982254], R² = [0.29399388 0.99911062]
Seed 90: NRMSE = [0.84832844 0.04368551], R² = [0.28033886 0.99809158]


In [8]:
print("Parallel Computing")

results = Parallel(n_jobs=-1, verbose=10)(delayed(seeded_trial)(seed) for seed in seeds)

print("\nSeed Results")
for seed, nrmse, r_2 in results[::10]:
    print(f"Seed {seed}: NRMSE = {nrmse}, R² = {r_2}")

Parallel Computing


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   13.1s
[Parallel(n_jobs=-1)]: Done  12 tasks      | elapsed:   15.8s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:   18.1s
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:   18.6s
[Parallel(n_jobs=-1)]: Done  41 tasks      | elapsed:   21.7s
[Parallel(n_jobs=-1)]: Done  52 tasks      | elapsed:   23.3s
[Parallel(n_jobs=-1)]: Done  65 tasks      | elapsed:   25.0s
[Parallel(n_jobs=-1)]: Done  78 tasks      | elapsed:   27.0s
[Parallel(n_jobs=-1)]: Done  92 out of 100 | elapsed:   29.4s remaining:    2.6s



Seed Results
Seed 0: NRMSE = [0.84961985 0.04403428], R² = [0.27814611 0.99806098]
Seed 10: NRMSE = [0.85448154 0.06008886], R² = [0.2698613  0.99638933]
Seed 20: NRMSE = [0.85304171 0.08463523], R² = [0.27231983 0.99283688]
Seed 30: NRMSE = [0.84546227 0.02583743], R² = [0.28519355 0.99933243]
Seed 40: NRMSE = [0.84929764 0.04147623], R² = [0.27869352 0.99827972]
Seed 50: NRMSE = [0.84213403 0.01713274], R² = [0.29081027 0.99970647]
Seed 60: NRMSE = [0.84447049 0.01510623], R² = [0.28686959 0.9997718 ]
Seed 70: NRMSE = [0.84462873 0.05503047], R² = [0.28660231 0.99697165]
Seed 80: NRMSE = [0.84024171 0.02982254], R² = [0.29399388 0.99911062]
Seed 90: NRMSE = [0.84832844 0.04368551], R² = [0.28033886 0.99809158]


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   29.9s finished


In [49]:
seed_42 = results[42]
seed_0_30 = results[0:31]

print("Comparison of Seed 42 with Seeds 1-30")
print(f"Seed 42: NRMSE = {seed_42[1]}, R² = {seed_42[2]}")
print(f"Seeds 1-30: NRMSE mean = {np.mean([r[1] for r in seed_0_30])}, R² mean = {np.mean([r[2] for r in seed_0_30])}")

Comparison of Seed 42 with Seeds 1-30
Seed 42: NRMSE = [0.84325412 0.02982294], R² = [0.28892249 0.99911059]
Seeds 1-30: NRMSE mean = 0.447482106271886, R² mean = 0.6381417165077032


In [86]:
# This will have the x nrmse and r2
seed_42_results_x = np.array(seed_42[1:])[::, 0]
seed_1_30_results_x = np.array([results[1:] for results in seed_0_30])[::, ::, 0]
print("Comparison of Seed 42 with Seeds 1-30 for henon in x Dim")
print(f"Seed 42: NRMSE = {seed_42_results_x[0]}, R² = {seed_42_results_x[1]}")
print(
    f"Seeds 1-30: NRMSE mean = {seed_1_30_results_x[::, 0].mean(axis=0)}, R² mean = {seed_1_30_results_x[::, 1].mean(axis=0)}"
)

Comparison of Seed 42 with Seeds 1-30 for henon in x Dim
Seed 42: NRMSE = 0.8432541187879252, R² = 0.2889224911471998
Seeds 1-30: NRMSE mean = 0.8492829566151405, R² mean = 0.27868970880031113


In [95]:
bg_results = [r for r in results if r[0] != 42]
seed_42_res = [r for r in results if r[0] == 42][0]

bg_nrmse_x = [r[1][0] for r in bg_results]
bg_nrmse_y = [r[1][1] for r in bg_results]
bg_r2_x = [r[2][0] for r in bg_results]
bg_r2_y = [r[2][1] for r in bg_results]

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Dimension X: NRMSE (Lower is Better)",
        "Dimension X: R² Score (Higher is Better)",
        "Dimension Y: NRMSE (Lower is Better)",
        "Dimension Y: R² Score (Higher is Better)",
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

plot_grid = [
    {
        "row": 1,
        "col": 1,
        "bg_data": bg_nrmse_x,
        "target_val": seed_42_res[1][0],
        "metric": "NRMSE",
    },
    {
        "row": 1,
        "col": 2,
        "bg_data": bg_r2_x,
        "target_val": seed_42_res[2][0],
        "metric": "R2",
    },
    {
        "row": 2,
        "col": 1,
        "bg_data": bg_nrmse_y,
        "target_val": seed_42_res[1][1],
        "metric": "NRMSE",
    },
    {
        "row": 2,
        "col": 2,
        "bg_data": bg_r2_y,
        "target_val": seed_42_res[2][1],
        "metric": "R2",
    },
]

colors = {"Background": "rgba(100, 149, 237, 0.6)", "Seed42": "red"}

for pane in plot_grid:
    row, col = pane["row"], pane["col"]
    show_legend = True if (row == 1 and col == 1) else False

    # Box
    fig.add_trace(
        go.Box(
            y=pane["bg_data"],
            name="Random Seeds",
            boxpoints="all",
            jitter=0.4,
            pointpos=-1.8,
            marker=dict(color=colors["Background"], size=4),
            line=dict(width=1.5),
            showlegend=show_legend,
        ),
        row=row,
        col=col,
    )

    # Seed 42
    fig.add_trace(
        go.Scatter(
            x=["Random Seeds"],
            y=[pane["target_val"]],
            mode="markers",
            marker=dict(
                color=colors["Seed42"],
                size=12,
                symbol="star",
                line=dict(width=1, color="black"),
            ),
            name="Seed 42",
            showlegend=show_legend,
        ),
        row=row,
        col=col,
    )

    fig.update_yaxes(title_text=pane["metric"], row=row, col=col)

fig.update_layout(
    title_text="Henon Map Reservoir Performance: 100 seeds vs seed 42",
    template="plotly_white",
    height=800,
    width=1000,
)

fig.show()

In [96]:
save_figure(
    fig=fig,
    filepath="./plots/2_hundred_seeds_vs_42",
)